# 07 — Train model experiments

**Project:** Hotel overbooking decision support  
**Stage:** Fit initial predictive models on Train; compare on Choose  
**Purpose:** Test whether selected booking-level features improve cancellation probabilities beyond the simple baselines.

This notebook fits a small set of understandable models:
1. A dummy model that predicts the Train cancellation rate for every booking.
2. Logistic regression.
3. A random forest.

The feature list follows Notebook 05. All preprocessing is fitted inside each model pipeline using Train data only. Choose is used for evaluation, not fitting.

**Do not use the Proof period here.** Model tuning and more detailed comparisons belong in later notebooks.

In [1]:
# Run from the repository root, so `from src import ...` and the relative evidence/ paths work
# whether this notebook is opened from the repo folder or from notebooks/pipeline/.
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "contract.py").exists())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Working directory:", REPO_ROOT)

Working directory: C:\Users\Lenovo\Desktop\Build_2_Repo\IIT_GN_W06


## 1. Load data and define the chronological periods

We derive arrival date and total stay nights for the evaluation contract. The original booking rows remain unchanged.

In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src import data, contract

raw = data.read_raw()
bookings = raw.copy()
bookings["arrival_date"] = data.arrival_date(bookings)
bookings["nights"] = (
    bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
)

train, choose, proof = contract.split_periods(bookings)

print("Train:", len(train), "rows")
print("Choose:", len(choose), "rows")
print("Proof:", len(proof), "rows")

Train: 49157 rows
Choose: 29546 rows
Proof: 40687 rows


## 2. Recreate the approved candidate feature table

The source code in `src/features.py` currently includes `country`, while Notebook 05 holds it back pending timing validation. To keep this experiment aligned with the notebook decision, we explicitly use the Notebook 05 candidate list below and do not call that module.

The feature timing caveat remains: these fields are not yet proven to be available at the original booking timestamp.

In [3]:
categorical_columns = [
    "hotel",
    "market_segment",
    "distribution_channel",
    "agent",
    "company"
]

numeric_columns = [
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled"
]

feature_columns = categorical_columns + numeric_columns

def prepare_features(frame):
    X = frame[feature_columns].copy()

    for column in categorical_columns:
        X[column] = X[column].astype("string")

    X["agent"] = X["agent"].fillna("NO_AGENT")
    X["company"] = X["company"].fillna("NO_COMPANY")

    for column in ["hotel", "market_segment", "distribution_channel"]:
        X[column] = X[column].fillna("UNKNOWN_" + column.upper())

    return X

X_train = prepare_features(train)
X_choose = prepare_features(choose)

y_train = train["is_canceled"].astype(int)
y_choose = choose["is_canceled"].astype(int)

print("Train feature shape:", X_train.shape)
print("Choose feature shape:", X_choose.shape)

Train feature shape: (49157, 8)
Choose feature shape: (29546, 8)


## 3. Define preprocessing

Categorical columns are one-hot encoded. A category not seen during Train is ignored by the encoder at prediction time rather than causing an error.

Numeric columns are standardised for logistic regression. Scaling is not essential for random forests, but using the same preprocessing structure keeps the experiment straightforward.

The transformer is part of each pipeline. It is fitted only when the pipeline is fitted on Train.

In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns
        ),
        (
            "numeric",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_columns
        )
    ],
    remainder="drop"
)

## 4. Define the three models

- **Dummy classifier:** the minimum benchmark; it predicts the observed Train class proportions.
- **Logistic regression:** a relatively simple linear model that estimates cancellation probability.
- **Random forest:** a collection of decision trees that can capture non-linear patterns and interactions.

We use fixed, modest settings here. Hyperparameter experimentation belongs in Notebook 08.

In [5]:
models = {
    "dummy": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", DummyClassifier(strategy="prior"))
    ]),
    "logistic_regression": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "random_forest": Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=150,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

print("Models:", list(models))

Models: ['dummy', 'logistic_regression', 'random_forest']


## 5. Fit on Train only

No Choose or Proof rows are passed to `.fit()`. The fitted pipelines include their own preprocessing steps.

In [6]:
for model_name, model in models.items():
    print("Fitting:", model_name)
    model.fit(X_train, y_train)

print("All models fitted on Train.")

Fitting: dummy


Fitting: logistic_regression


Fitting: random_forest


All models fitted on Train.


## 6. Predict cancellation probabilities on Choose

The positive class is `1` (cancelled or no-show). We select its probability column explicitly rather than assuming a column position.

In [7]:
choose_probabilities = pd.DataFrame(index=choose.index)

for model_name, model in models.items():
    class_values = list(model.named_steps["model"].classes_)
    positive_class_position = class_values.index(1)

    choose_probabilities[model_name] = model.predict_proba(
        X_choose
    )[:, positive_class_position]

display(choose_probabilities.head())

assert choose_probabilities.notna().all().all()
assert choose_probabilities.apply(
    lambda column: column.between(0, 1).all()
).all()

,dummy,logistic_regression,random_forest
6765,0.357731,0.295889,0.282691
6766,0.357731,0.295889,0.282691
6767,0.357731,0.292715,0.274009
6768,0.357731,0.295889,0.282691
6769,0.357731,0.139013,0.160512


## 7. Evaluate booking-level probability quality

We report:
- **Brier score:** lower is better.
- **Log loss:** lower is better.
- **ROC AUC:** higher is better; it measures ranking, not probability calibration.

A model can rank bookings well and still produce poorly calibrated probabilities. Calibration is examined separately in Notebook 09.

In [8]:
booking_scores = []

for model_name in models:
    probabilities = choose_probabilities[model_name]

    booking_scores.append({
        "model": model_name,
        "brier_score": brier_score_loss(y_choose, probabilities),
        "log_loss": log_loss(y_choose, probabilities, labels=[0, 1]),
        "roc_auc": roc_auc_score(y_choose, probabilities)
    })

booking_scores = (
    pd.DataFrame(booking_scores)
    .sort_values("brier_score")
    .reset_index(drop=True)
)

display(booking_scores)

,model,brier_score,log_loss,roc_auc
0,random_forest,0.210250,0.607264,0.690719
1,logistic_regression,0.210995,0.617134,0.699128
2,dummy,0.232884,0.658529,0.500000


## 8. Evaluate expected cancellations by hotel-arrival night

The client's operational unit is one hotel on one arrival night. We sum each booking's predicted cancellation probability to estimate the expected number of cancellations that night.

We compare this expected count with the actual cancellation count. This is not a room-capacity calculation and does not estimate actual guest walks.

In [9]:
night_scores = []

for model_name in models:
    scored = choose[["hotel", "arrival_date", "is_canceled"]].copy()
    scored["probability"] = choose_probabilities[model_name].values

    by_night = scored.groupby(["hotel", "arrival_date"]).agg(
        actual_cancellations=("is_canceled", "sum"),
        expected_cancellations=("probability", "sum")
    )

    night_mae = (
        by_night["actual_cancellations"]
        - by_night["expected_cancellations"]
    ).abs().mean()

    night_scores.append({
        "model": model_name,
        "hotel_arrival_nights": len(by_night),
        "mean_absolute_error_cancellations": night_mae
    })

night_scores = (
    pd.DataFrame(night_scores)
    .sort_values("mean_absolute_error_cancellations")
    .reset_index(drop=True)
)

display(night_scores)

,model,hotel_arrival_nights,mean_absolute_error_cancellations
0,random_forest,368,8.878393
1,dummy,368,9.661821
2,logistic_regression,368,9.911024


## 9. Evaluate each hotel separately

The group-level score can hide different results between the two hotels. This table reports booking-level Brier score for each property.

In [10]:
hotel_scores = []

for hotel_name, hotel_rows in choose.groupby("hotel"):
    row_index = hotel_rows.index
    actual = y_choose.loc[row_index]

    for model_name in models:
        hotel_scores.append({
            "hotel": hotel_name,
            "model": model_name,
            "brier_score": brier_score_loss(
                actual,
                choose_probabilities.loc[row_index, model_name]
            )
        })

hotel_scores = pd.DataFrame(hotel_scores)

display(
    hotel_scores.pivot(
        index="model",
        columns="hotel",
        values="brier_score"
    )
)

hotel,City Hotel,Resort Hotel
model,,
dummy,0.243990,0.208885
logistic_regression,0.218878,0.193960
random_forest,0.218866,0.191632


## 10. Compare with the simple baselines

Notebook 06 compared global, hotel-specific and partially pooled rates. This cell recreates those baselines using Train and scores them on the same Choose rows.

The comparison is fair only because every method is evaluated on the same Choose bookings.

In [11]:
global_rate = y_train.mean()
hotel_rates = train.groupby("hotel")["is_canceled"].mean()

baseline_probabilities = pd.DataFrame(index=choose.index)
baseline_probabilities["global_rate"] = global_rate
baseline_probabilities["hotel_rate"] = choose["hotel"].map(hotel_rates)

for strength in [20, 50, 100]:
    summary = train.groupby("hotel")["is_canceled"].agg(
        bookings="size",
        cancellations="sum"
    )
    pooled = (
        summary["cancellations"] + strength * global_rate
    ) / (
        summary["bookings"] + strength
    )
    baseline_probabilities[f"pooled_m_{strength}"] = choose["hotel"].map(pooled)

all_probabilities = pd.concat(
    [baseline_probabilities, choose_probabilities],
    axis=1
)

comparison = pd.DataFrame([
    {
        "method": method,
        "brier_score": brier_score_loss(y_choose, all_probabilities[method]),
        "log_loss": log_loss(y_choose, all_probabilities[method], labels=[0, 1])
    }
    for method in all_probabilities.columns
]).sort_values("brier_score").reset_index(drop=True)

display(comparison)

,method,brier_score,log_loss
0,random_forest,0.210250,0.607264
1,logistic_regression,0.210995,0.617134
2,pooled_m_100,0.229853,0.651993
3,pooled_m_50,0.229860,0.652012
4,pooled_m_20,0.229864,0.652023
5,hotel_rate,0.229867,0.652031
6,global_rate,0.232884,0.658529
7,dummy,0.232884,0.658529


## 11. Record the provisional result

Do not select a final model from one metric alone. Consider:
- probability quality (Brier and log loss);
- ranking ability (ROC AUC);
- nightly cancellation-count error;
- performance for each hotel;
- complexity and ease of explaining the result;
- whether the feature timing assumptions can be defended.

The next notebooks will examine hyperparameters, calibration and uncertainty. The final operational policy is selected later using Choose-period evidence.

In [12]:
print("Best booking-level Brier result:", comparison.iloc[0]["method"])
print("Best nightly cancellation-count MAE:", night_scores.iloc[0]["model"])
print()
print("These are provisional experiment leaders, not an approved production model.")

Best booking-level Brier result: random_forest
Best nightly cancellation-count MAE: random_forest

These are provisional experiment leaders, not an approved production model.


## 12. Save the experiment evidence

The model pipelines are not saved as production artifacts here. This notebook records the evaluation tables so the results can be reviewed and reproduced.

In [13]:
evidence_folder = Path("evidence/train_model_experiments")
evidence_folder.mkdir(parents=True, exist_ok=True)

choose_probabilities.to_csv(
    evidence_folder / "choose_model_probabilities.csv",
    index=True
)
booking_scores.to_csv(
    evidence_folder / "choose_booking_metrics.csv",
    index=False
)
night_scores.to_csv(
    evidence_folder / "choose_nightly_metrics.csv",
    index=False
)
hotel_scores.to_csv(
    evidence_folder / "choose_hotel_metrics.csv",
    index=False
)
comparison.to_csv(
    evidence_folder / "choose_baseline_model_comparison.csv",
    index=False
)

print("Saved model experiment evidence to:", evidence_folder)

Saved model experiment evidence to: evidence\train_model_experiments


## 13. Limitations and questions

### Limitations

- The feature list is provisional because the source export is a near-arrival snapshot.
- The current models are initial candidates, not tuned or calibrated.
- The dummy classifier is a benchmark, not a useful decision model.
- Choose-period performance is retrospective and must not be confused with a genuinely prospective test.
- The data does not include hotel room capacity, room-level inventory, rooms per booking or actual walk outcomes.
- A better booking-level score does not automatically imply a better overbooking policy.

### Questions

1. Why must the preprocessor be fitted only on Train?
2. What does `handle_unknown="ignore"` do?
3. What is the difference between Brier score and ROC AUC?
4. Why do we sum probabilities to estimate nightly cancellations?
5. Why is the best predictive model not necessarily the best operating policy?

**Next notebook:** `08_hyperparameter_experiments.ipynb`, where we test a controlled set of model settings without touching the Proof period.